# Controlled DAC versus UniDAC GPU benchmark

This notebook measures both metric-depth models in one Colab runtime. It uses the same decoded G1_A frames and GPU, excludes image/model loading from the timed region, performs five warm-up runs per model, and records ten repetitions per frame. The results are written to Google Drive as CSV, JSON, and Markdown.

## 1. Select a GPU runtime

Choose **Runtime → Change runtime type → GPU**. The dependency cell restarts the kernel once; after reconnection choose **Runtime → Run all** again.

In [ ]:
import subprocess
import sys
from pathlib import Path

setup_marker = Path("/content/.dac_unidac_benchmark_numpy_1_26_4_v1_ready")
packages = [
    "numpy==1.26.4",
    "opencv-python==4.11.0.86",
    "einops>=0.6",
    "timm>=0.9",
    "huggingface_hub>=0.20",
    "matplotlib>=3.8",
    "scipy>=1.10",
]

if not setup_marker.exists():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", *packages]
    )
    setup_marker.write_text("numpy==1.26.4\n")
    print(
        "Dependencies installed. Colab is restarting once. After it "
        "reconnects, choose Runtime → Run all."
    )
    import IPython

    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("Pinned benchmark dependencies are ready.")

In [ ]:
from pathlib import Path

PROJECT_REPO_URL = "https://github.com/esthy13/monocular-depth-estimation.git"
PROJECT_REF = "main"
PROJECT_REPO_IS_PRIVATE = True
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
DAC_COMMIT = "371ee299429257bb9a27d1e23b7dc53670e37023"
UNIDAC_COMMIT = "9ddfc1f4cea68e08273ec9bca037f2ef9e1aa90e"
DAC_VARIANT = "dac-indoor-resnet101"
DAC_FORWARD_SIZE = (500, 750)

PROJECT_DIR = Path("/content/monocular-depth-estimation")
DAC_DIR = PROJECT_DIR / "third_party" / "depth_any_camera"
UNIDAC_DIR = PROJECT_DIR / "third_party" / "UniDAC"
DATA_DIR = Path("/content/drive/MyDrive/cv_project_data")
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/cv_project_outputs/benchmark/recording1"
)
CACHE_DIR = Path("/content/drive/MyDrive/cv_project_cache")

RECORDING = "recording1"
SENSOR = "G1_A"
FRAME_INDICES = [0, 15, 30, 45, 60, 75, 90, 105, 120, 132]
WARMUP_RUNS = 5
TIMED_RUNS_PER_FRAME = 10
MODEL_ORDER = ["DAC", "UniDAC"]
RUN_BENCHMARK = True

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Choose Runtime → Change runtime type → GPU, "
        "then reconnect and run all cells again."
    )
if not (DATA_DIR / "intrinsic.json").is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_DIR / 'intrinsic.json'}. Upload cv_project_data "
        "to Drive or edit DATA_DIR."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}")
print(f"Output: {OUTPUT_DIR}")

## 2. Fetch pinned source

For the private project repository, add an expiring GitHub token with read-only **Contents** access to Colab Secrets as `GITHUB_TOKEN`, then enable notebook access. The token is passed through an HTTP header and is never written into the clone URL or notebook output.

In [ ]:
import base64
import os
import shutil
import subprocess
import tarfile
import urllib.request

from google.colab import userdata

def project_git_environment():
    environment = os.environ.copy()
    if not PROJECT_REPO_IS_PRIVATE:
        return environment
    try:
        github_token = userdata.get(GITHUB_TOKEN_SECRET)
    except Exception as error:
        raise RuntimeError(
            f"Add a GitHub token named {GITHUB_TOKEN_SECRET!r} in Colab "
            "Secrets, enable notebook access, and rerun this cell."
        ) from error
    if not github_token:
        raise RuntimeError(f"Colab Secret {GITHUB_TOKEN_SECRET!r} is empty.")
    credential = base64.b64encode(
        f"x-access-token:{github_token}".encode()
    ).decode()
    environment.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}",
    })
    return environment

git_environment = project_git_environment()

def run_git(arguments):
    subprocess.check_call([str(item) for item in arguments], env=git_environment)

if not (PROJECT_DIR / ".git").is_dir():
    run_git([
        "git", "clone", "--branch", PROJECT_REF, "--single-branch",
        PROJECT_REPO_URL, PROJECT_DIR,
    ])
else:
    run_git(["git", "-C", PROJECT_DIR, "fetch", "origin", PROJECT_REF])
    run_git(["git", "-C", PROJECT_DIR, "checkout", PROJECT_REF])
    run_git([
        "git", "-C", PROJECT_DIR, "merge", "--ff-only",
        f"origin/{PROJECT_REF}",
    ])

def install_pinned_archive(target, repository, commit, archive_name):
    marker = target / ".pinned_commit"
    installed = marker.read_text().strip() if marker.is_file() else None
    if target.exists() and installed != commit:
        raise RuntimeError(
            f"{target} is not the requested pinned source. Start a fresh "
            "runtime and rerun the notebook."
        )
    if target.exists():
        return
    archive_path = Path("/content") / archive_name
    staging = Path("/content") / f"{archive_name}-source"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    urllib.request.urlretrieve(
        f"https://github.com/{repository}/archive/{commit}.tar.gz",
        archive_path,
    )
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(staging)
    extracted = next(path for path in staging.iterdir() if path.is_dir())
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extracted), str(target))
    marker.write_text(commit + "\n")
    archive_path.unlink(missing_ok=True)
    shutil.rmtree(staging)

install_pinned_archive(
    DAC_DIR, "yuliangguo/depth_any_camera", DAC_COMMIT, "dac-source.tar.gz"
)
install_pinned_archive(
    UNIDAC_DIR, "girish1511/UniDAC", UNIDAC_COMMIT, "unidac-source.tar.gz"
)
project_commit = subprocess.check_output(
    ["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True
).strip()
git_environment.pop("GIT_CONFIG_VALUE_0", None)
print(f"Project commit: {project_commit}")
print(f"DAC commit:     {DAC_COMMIT}")
print(f"UniDAC commit:  {UNIDAC_COMMIT}")

In [ ]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from huggingface_hub import hf_hub_download
from src.depth_models import DepthAnyCamera

dac_cache = CACHE_DIR / "dac"
unidac_cache = CACHE_DIR / "unidac"
config_file, checkpoint_file = DepthAnyCamera.VARIANTS[DAC_VARIANT]
dac_config = Path(hf_hub_download(
    repo_id=DepthAnyCamera.HF_REPO, filename=config_file,
    local_dir=str(dac_cache),
))
dac_checkpoint = Path(hf_hub_download(
    repo_id=DepthAnyCamera.HF_REPO, filename=checkpoint_file,
    local_dir=str(dac_cache),
))
unidac_checkpoint = Path(hf_hub_download(
    repo_id="girish1511/UniDAC", filename="unidac.pt",
    local_dir=str(unidac_cache),
))
print(f"DAC checkpoint: {dac_checkpoint}")
print(f"UniDAC checkpoint: {unidac_checkpoint}")

## 3. Run the benchmark

The script loads DAC, benchmarks it, releases its GPU memory, then does the same for UniDAC. Ten frames × ten repetitions produce 100 latency measurements per model. Do not run other GPU work in this runtime while it is measuring.

In [ ]:
from IPython.display import Markdown, display

if not RUN_BENCHMARK:
    print("Benchmark is disabled. Set RUN_BENCHMARK=True when ready.")
else:
    command = [
        sys.executable,
        str(PROJECT_DIR / "benchmark_depth_models.py"),
        "--data_dir", str(DATA_DIR),
        "--output_dir", str(OUTPUT_DIR),
        "--recording", RECORDING,
        "--sensor", SENSOR,
        "--frame_indices", *[str(index) for index in FRAME_INDICES],
        "--warmup_runs", str(WARMUP_RUNS),
        "--timed_runs", str(TIMED_RUNS_PER_FRAME),
        "--model_order", *MODEL_ORDER,
        "--dac_repo_dir", str(DAC_DIR),
        "--dac_config_path", str(dac_config),
        "--dac_checkpoint_path", str(dac_checkpoint),
        "--dac_variant", DAC_VARIANT,
        "--dac_forward_size", *[str(value) for value in DAC_FORWARD_SIZE],
        "--unidac_repo_dir", str(UNIDAC_DIR),
        "--unidac_checkpoint_path", str(unidac_checkpoint),
    ]
    subprocess.check_call(command, cwd=PROJECT_DIR)
    report_path = OUTPUT_DIR / "benchmark_report.md"
    display(Markdown(report_path.read_text()))
    print(f"Download this folder after completion: {OUTPUT_DIR}")